# 🏦 保险保单受益人变更检索 PoC## 业务背景保险公司持有**大量多页 TIF 格式的历史保单**，当发生受益人变更时，需要快速从海量保单中定位到：1. **哪份保单**发生了变更2. **变更发生的位置**（第几页、第几段）3. **变更前后内容对比**## 系统架构```[多页TIF保单] --> [OSS存储] --> [异步ETL解析]                                 |            ---------------------+--------------------            v                    v                    v     [视觉布局分析]      [元数据提取]          [父子分块]     (OCR+Markdown)    (保单号/投保人)         (Chunking)                                 |            ---------------------+            v     [混合存储与检索]           [BM25全文索引]     [向量数据库]              [关键词检索]            |            v     [混合重排层] --> Metadata Filter + Reranker            |            v     [LLM生成] --> 最终答案```## 📋 PoC 目标本 Notebook 使用 **Mock 数据** 完整演示上述管道，回答：> *"保单 P0242025-1883 的受益人是否有过变更？变更内容是什么？"*

---## 1. 环境准备 & 依赖安装

In [ ]:
# @title 安装依赖（首次运行需执行）import sys, subprocess, importlib, importlib.util_required = ['numpy', 'pandas', 'scikit-learn', 'jieba', 'rank_bm25', 'sentence-transformers', 'tabulate']_missing = []for p in _required:    spec = importlib.util.find_spec(p.replace('-', '_').replace('.', '_'))    if spec is None:        _missing.append(p)if _missing:    print(f"Install missing: {_missing}")    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _missing)print('Dependencies OK')

In [ ]:
# @title 导入模块import json, re, copy, textwrap, io, contextlibfrom dataclasses import dataclass, fieldfrom typing import List, Optional, Dict, Tuplefrom uuid import uuid4from datetime import datetime, timedeltaimport numpy as npimport pandas as pdfrom sklearn.preprocessing import normalizeimport jiebafrom rank_bm25 import BM25Okapifrom tabulate import tabulateprint('Imports OK')

---## 2. Mock 数据：模拟多页 TIF 保单在实际场景中，TIF 文件经过 OCR + 视觉布局分析（如 PPOCR、LayoutLM）转换为 Markdown 文本。这里我们直接模拟解析后的结果——**3 份保单**，每份多页，其中包含受益人变更记录。

In [ ]:
# @title 模拟保单数据np.random.seed(42)def _make_page(pn, text):    return {"page_number": pn, "page_image": f"mock_tif/policy_{pn:04d}.tif", "markdown": text, "ocr_confidence": round(0.92 + 0.05 * np.random.random(), 2)}P1 = "## 保险合同\n**保单号**: P0242025-1883\n**投保人**: 张建国\n**被保险人**: 张小明\n**险种**: 国寿鑫享金生年金保险（A 款）\n**保费**: 12,800.00/年\n**保单生效日**: 2020-03-15"P2 = "## 保险责任\n\n### 年金给付\n自本合同生效之日起，被保险人生存至第五个保单周年日，我们按本合同基本保险金额的 100% 给付年金。\n\n### 身故保险金\n被保险人身故，我们按本合同已交保险费减去累计已给付年金后的余额与现金价值的较大者给付身故保险金。"P3 = "## 受益人\n\n### 身故保险金受益人\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 李芳 | 100% | 第一顺序 | 配偶 |\n\n> 备注：若受益人先于被保险人身故，则身故保险金作为被保险人的遗产处理。"P4 = "## 受益人变更批单\n**批单号**: BG2024-00321\n**变更日期**: 2024-06-20\n**变更类型**: 受益人变更\n\n### 变更前\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 李芳 | 100% | 第一顺序 | 配偶 |\n\n### 变更后\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 张美玲 | 60% | 第一顺序 | 女儿 |\n| 李芳 | 40% | 第一顺序 | 配偶 |\n\n**变更原因**: 增加子女为共同受益人"P5 = "## 缴费记录\n\n| 缴费年度 | 缴费日期 | 缴费金额 | 状态 |\n|----------|----------|----------|------|\n| 第1年 | 2020-03-15 | 12,800.00 | 已缴 |\n| 第2年 | 2021-03-15 | 12,800.00 | 已缴 |\n| 第3年 | 2022-03-15 | 12,800.00 | 已缴 |\n| 第4年 | 2023-03-15 | 12,800.00 | 已缴 |\n| 第5年 | 2024-03-15 | 12,800.00 | 已缴 |"Q1 = "## 保险合同\n**保单号**: P2024-66892\n**投保人**: 王秀英\n**被保险人**: 王秀英\n**险种**: 平安福2024版\n**保费**: 6,500.00/年\n**保单生效日**: 2024-01-10"Q2 = "## 保险责任\n\n### 重大疾病保险金\n经医院确诊初次发生本合同约定的120种重大疾病，我们按基本保险金额的100%给付重大疾病保险金。\n\n### 轻症保险金\n经医院确诊初次发生本合同约定的40种轻症，我们按基本保险金额的20%给付轻症保险金，累计限3次。"Q3 = "## 受益人\n\n### 身故保险金受益人\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 赵强 | 100% | 第一顺序 | 配偶 |\n\n### 满期保险金受益人\n本合同满期保险金的受益人为被保险人本人。"Q4 = "## 免责条款\n\n因下列情形之一导致被保险人身故的，我们不承担给付身故保险金的责任：\n1. 投保人对被保险人的故意杀害、故意伤害；\n2. 被保险人故意犯罪或者抗拒依法采取的刑事强制措施；\n3. 被保险人自本合同成立之日起2年内自杀；"R1 = "## 保险合同\n**保单号**: TPK-2023-004517\n**投保人**: 刘伟\n**被保险人**: 刘小宇\n**险种**: 金佑人生终身寿险\n**保费**: 9,200.00/年\n**保单生效日**: 2023-07-01"R2 = "## 受益人\n\n### 身故保险金受益人\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 陈静 | 100% | 第一顺序 | 配偶 |"R3 = "## 受益人变更批单（第一次）\n**批单号**: BG2024-00892\n**变更日期**: 2024-03-10\n**变更类型**: 受益人变更\n\n### 变更前\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 陈静 | 100% | 第一顺序 | 配偶 |\n\n### 变更后\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 陈静 | 50% | 第一顺序 | 配偶 |\n| 刘伟 | 50% | 第一顺序 | 父亲 |\n\n**变更原因**: 增加投保人为共同受益人"R4 = "## 受益人变更批单（第二次）\n**批单号**: BG2025-00103\n**变更日期**: 2025-01-15\n**变更类型**: 受益人变更\n\n### 变更前\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 陈静 | 50% | 第一顺序 | 配偶 |\n| 刘伟 | 50% | 第一顺序 | 父亲 |\n\n### 变更后\n| 姓名 | 受益比例 | 受益顺序 | 与被保险人关系 |\n|------|----------|----------|---------------|\n| 陈静 | 40% | 第一顺序 | 配偶 |\n| 刘伟 | 30% | 第一顺序 | 父亲 |\n| 刘晓梅 | 30% | 第一顺序 | 姐姐 |\n\n**变更原因**: 增加子女为共同受益人"policy_a = {"policy_id": "P0242025-1883", "insurer": "中国人寿", "pages": [_make_page(1, P1), _make_page(2, P2), _make_page(3, P3), _make_page(4, P4), _make_page(5, P5)]}policy_b = {"policy_id": "P2024-66892", "insurer": "中国平安", "pages": [_make_page(1, Q1), _make_page(2, Q2), _make_page(3, Q3), _make_page(4, Q4)]}policy_c = {"policy_id": "TPK-2023-004517", "insurer": "太平洋人寿", "pages": [_make_page(1, R1), _make_page(2, R2), _make_page(3, R3), _make_page(4, R4)]}POLICIES = [policy_a, policy_b, policy_c]print(f'Generated {len(POLICIES)} policies, {sum(len(p["pages"]) for p in POLICIES)} pages total')

In [ ]:
# @title 查看模拟数据概览rows = []for p in POLICIES:    for page in p['pages']:        rows.append({'policy': p['policy_id'], 'insurer': p['insurer'], 'page': page['page_number'], 'ocr_conf': page['ocr_confidence'], 'preview': page['markdown'][:80].replace(chr(10), ' | ')})df_pages = pd.DataFrame(rows)print(tabulate(df_pages, headers='keys', tablefmt='grid', showindex=False, maxcolwidths=[20, 14, 6, 10, 55]))print(f'\nTotal: {len(POLICIES)} policies, {len(df_pages)} pages')

In [ ]:
# @title 元数据提取器@dataclassclass PolicyMetadata:    policy_id: str    insurer: str    applicant: Optional[str] = None    insured: Optional[str] = None    product_name: Optional[str] = None    premium: Optional[str] = None    effective_date: Optional[str] = None    beneficiaries: List[Dict] = field(default_factory=list)    endorsements: List[Dict] = field(default_factory=list)def extract_policy_metadata(policy):    full_text = chr(10).join(p['markdown'] for p in policy['pages'])    meta = PolicyMetadata(policy_id=policy['policy_id'], insurer=policy['insurer'])    for pat, attr in [        (r'\*\*投保人\*\*:\s*(.+)', 'applicant'),        (r'\*\*被保险人\*\*:\s*(.+)', 'insured'),        (r'\*\*险种\*\*:\s*(.+)', 'product_name'),        (r'\*\*保费\*\*:\s*(.+)', 'premium'),        (r'\*\*保单生效日\*\*:\s*(.+)', 'effective_date'),    ]:        m = re.search(pat, full_text)        if m: setattr(meta, attr, m.group(1).strip())    in_ben = False    for page in policy['pages']:        for line in page['markdown'].split(chr(10)):            if re.search(r'(?<!变更)(?:身故保险金|受益人)', line) and '|' not in line:                in_ben = True            if in_ben and line.startswith('|'):                cells = [c.strip() for c in line.split('|') if c.strip()]                if len(cells) >= 4 and cells[0] not in ('姓名', '------'):                    meta.beneficiaries.append({'name': cells[0], 'ratio': cells[1], 'order': cells[2], 'relation': cells[3]})            if in_ben and line.startswith('##') and '受益人' not in line:                in_ben = False    cur = {}    in_endo = False    in_b4, in_af = False, False    for page in policy['pages']:        for line in page['markdown'].split(chr(10)):            if '批单号' in line and '变更' not in line:                if cur and 'before' in cur:                    meta.endorsements.append(cur)                m = re.search(r'\*{0,2}批单号\s*[：:]\s*(.+)', line)                cur = {'endorsement_id': m.group(1).strip() if m else ''}                in_endo = True; in_b4 = in_af = False            if in_endo:                m = re.search(r'\*{0,2}变更日期\s*[：:]\s*(.+)', line)                if m: cur['date'] = m.group(1).strip()                m = re.search(r'\*{0,2}变更原因\s*[：:]\s*(.+)', line)                if m: cur['reason'] = m.group(1).strip()                if '### 变更前' in line: in_b4 = True; in_af = False; cur['before'] = []                if '### 变更后' in line: in_b4 = False; in_af = True; cur['after'] = []                if in_b4 and line.startswith('|'):                    cells = [c.strip() for c in line.split('|') if c.strip()]                    if len(cells) >= 4 and cells[0] not in ('姓名', '------'):                        cur['before'].append({'name': cells[0], 'ratio': cells[1], 'order': cells[2], 'relation': cells[3]})                if in_af and line.startswith('|'):                    cells = [c.strip() for c in line.split('|') if c.strip()]                    if len(cells) >= 4 and cells[0] not in ('姓名', '------'):                        cur['after'].append({'name': cells[0], 'ratio': cells[1], 'order': cells[2], 'relation': cells[3]})    if cur and 'before' in cur:        meta.endorsements.append(cur)    return metaall_metadata = {p['policy_id']: extract_policy_metadata(p) for p in POLICIES}for pid, meta in all_metadata.items():    print(f'\n{"="*60}')    print(f'Policy: {pid} | {meta.insurer}')    print(f'  Applicant: {meta.applicant}')    print(f'  Insured: {meta.insured}')    print(f'  Product: {meta.product_name}')    print(f'  Current beneficiaries: {len(meta.beneficiaries)}')    for b in meta.beneficiaries:        print(f'    - {b["name"]} ({b["relation"]}): {b["ratio"]}')    print(f'  Endorsements: {len(meta.endorsements)}')    for e in meta.endorsements:        print(f'    - {e.get("endorsement_id", "")} ({e.get("date", "")}): {e.get("reason", "")}')

---## 3. 父子分块 (Parent-Child Chunking)将多页保单拆分为层级化的文本块：- **Parent Chunk**（父块）：一个完整章节- **Child Chunk**（子块）：父块中的小段落/表格检索时用子块找相似，返回时附上父块作为上下文。

In [ ]:
# @title 父子分块实现@dataclassclass Chunk:    chunk_id: str    policy_id: str    page_number: int    chunk_type: str    parent_id: Optional[str] = None    heading: str = ''    text: str = ''    metadata: Dict = field(default_factory=dict)def chunk_policy(policy):    chunks = []    for page in policy['pages']:        lines = page['markdown'].split(chr(10))        cur_parent = None        parent_lines = []        for line in lines:            hm = re.match(r'^(#{2,4})\s+(.+)', line)            if hm:                if cur_parent:                    cur_parent.text = chr(10).join(parent_lines).strip()                    chunks.append(cur_parent)                pid = str(uuid4())                cur_parent = Chunk(chunk_id=pid, policy_id=policy['policy_id'], page_number=page['page_number'], chunk_type='parent', heading=hm.group(2).strip(), text='', metadata={'level': len(hm.group(1))})                parent_lines = [line]                chunks.append(Chunk(chunk_id=str(uuid4()), policy_id=policy['policy_id'], page_number=page['page_number'], chunk_type='child', parent_id=pid, heading=hm.group(2).strip(), text=line, metadata={'level': len(hm.group(1)), 'is_heading': True}))            elif cur_parent:                parent_lines.append(line)                if line.strip() and not line.startswith('#'):                    if line.startswith('|') and chunks and chunks[-1].chunk_type == 'child' and chunks[-1].parent_id == cur_parent.chunk_id and chunks[-1].text.startswith('|'):                        chunks[-1].text += chr(10) + line                        continue                    chunks.append(Chunk(chunk_id=str(uuid4()), policy_id=policy['policy_id'], page_number=page['page_number'], chunk_type='child', parent_id=cur_parent.chunk_id, heading=cur_parent.heading, text=line, metadata={'is_heading': False}))        if cur_parent:            cur_parent.text = chr(10).join(parent_lines).strip()            chunks.append(cur_parent)    return chunksall_chunks = []for p in POLICIES:    all_chunks.extend(chunk_policy(p))parents = [c for c in all_chunks if c.chunk_type == 'parent']children = [c for c in all_chunks if c.chunk_type == 'child']print(f'Total chunks: {len(all_chunks)} (Parent: {len(parents)}, Child: {len(children)})')print('\nParent Chunk samples:')for p in parents[:3]:    print(f'  [{p.policy_id}] p{p.page_number} > {p.heading} ({len(p.text)} chars)')

---## 4. 混合检索 (Hybrid Retrieval)### 4.1 向量检索 (Dense Retrieval)使用 Sentence-BERT 将子块编码为向量，用余弦相似度检索 Top-K。> ⚠️ 此处使用 `all-MiniLM-L6-v2`（轻量英文模型）模拟。生产环境应使用中文专用模型如 `BAAI/bge-large-zh-v1.5`。

In [ ]:
# @title 构建向量索引from sentence_transformers import SentenceTransformerprint('Loading embedding model...')embedder = SentenceTransformer('all-MiniLM-L6-v2')child_texts = [c.text for c in children]child_embeddings = embedder.encode(child_texts, show_progress_bar=True, normalize_embeddings=True)print(f'Vector index built: {len(child_embeddings)} vectors, dim {child_embeddings.shape[1]}')

In [ ]:
# @title 向量检索函数def dense_search(query, top_k=10):    q_vec = embedder.encode([query], normalize_embeddings=True)[0]    scores = child_embeddings @ q_vec    top_indices = np.argsort(scores)[::-1][:top_k]    return [(children[i], float(scores[i])) for i in top_indices]test_query = '受益人变更'dense_results = dense_search(test_query, top_k=5)print(f'Dense search: "{test_query}"\n')for chunk, score in dense_results:    txt = (chunk.text[:80] + '...') if len(chunk.text) > 80 else chunk.text    print(f'  [{chunk.policy_id}] p{chunk.page_number} | score: {score:.4f} | {txt}')

### 4.2 BM25 全文检索 (Sparse Retrieval)使用 BM25 算法做关键词匹配检索，弥补向量检索对稀有/精确关键词的不足。

In [ ]:
# @title 构建 BM25 索引def tokenize(text):    tokens = []    for word in jieba.cut(text):        word = word.strip()        if word and word not in ('', ' ', chr(10), '|', '---', '**:'):            tokens.append(word.lower())    return tokenstokenized_corpus = [tokenize(c.text) for c in children]bm25 = BM25Okapi(tokenized_corpus)print(f'BM25 index built: {len(tokenized_corpus)} documents')def bm25_search(query, top_k=10):    q_tokens = tokenize(query)    scores = bm25.get_scores(q_tokens)    top_indices = np.argsort(scores)[::-1][:top_k]    return [(children[i], float(scores[i])) for i in top_indices if scores[i] > 0]bm25_results = bm25_search('张美玲 受益人 比例 60%', top_k=5)print(f'BM25 search: "张美玲 受益人 比例 60%"\n')for chunk, score in bm25_results:    txt = (chunk.text[:80] + '...') if len(chunk.text) > 80 else chunk.text    print(f'  [{chunk.policy_id}] p{chunk.page_number} | score: {score:.1f} | {txt}')

---## 5. 混合重排 (Hybrid Re-ranking)1. **合并向量 + BM25 结果**（Reciprocal Rank Fusion）2. **Metadata 过滤**：根据保单号精确过滤3. **Reranker 重排**：对候选块重新打分

In [ ]:
# @title 混合检索与重排def hybrid_retrieve(query, policy_id=None, top_k_dense=20, top_k_bm25=20, top_k_final=5, alpha=0.5):    dense_results = dense_search(query, top_k=top_k_dense)    bm25_results = bm25_search(query, top_k=top_k_bm25)    k_val = 60    fusion = {}    for rank, (chunk, _) in enumerate(dense_results):        fusion[chunk.chunk_id] = fusion.get(chunk.chunk_id, 0.0) + (alpha / (rank + k_val))    for rank, (chunk, _) in enumerate(bm25_results):        fusion[chunk.chunk_id] = fusion.get(chunk.chunk_id, 0.0) + ((1 - alpha) / (rank + k_val))    cmap = {c.chunk_id: c for c in children}    filtered = [(cmap[cid], sc) for cid, sc in fusion.items() if not policy_id or cmap[cid].policy_id == policy_id]    filtered.sort(key=lambda x: -x[1])    return filtered[:top_k_final]query = '受益人变更 比例'; target = 'P0242025-1883'final_results = hybrid_retrieve(query, policy_id=target, top_k_final=5)print(f'Hybrid search: "{query}" @ [{target}]\n')for chunk, score in final_results:    parent = next((p for p in parents if p.chunk_id == chunk.parent_id), None)    ctx = f' > Parent: {parent.heading}' if parent else ''    txt = (chunk.text[:100] + '...') if len(chunk.text) > 100 else chunk.text    print(f'  [p{chunk.page_number}]{ctx} | RRF: {score:.4f} | {txt}')

In [ ]:
# @title 检索效果对比query = '张美玲 受益比例 60% 变更'; target = 'P0242025-1883'dense_only = [(c, s) for c, s in dense_search(query, top_k=5) if c.policy_id == target]bm25_only = [(c, s) for c, s in bm25_search(query, top_k=5) if c.policy_id == target]hybrid = hybrid_retrieve(query, policy_id=target, top_k_final=5)print(f'{"="*60}')print(f'  Comparison: "{query}" @ [{target}]')print(f'{"="*60}')for method, results in [('Dense only', dense_only), ('BM25 only', bm25_only), ('Hybrid (RRF)', hybrid)]:    print(f'\n  {method}:')    if not results: print('    (no results)')    for i, (chunk, score) in enumerate(results):        marker = 'HIT' if ('变更' in chunk.text or '批单' in chunk.text) else '   '        txt = chunk.text[:70].strip()        print(f'    {marker} #{i+1} p{chunk.page_number} | score={score:.4f} | {txt}')

---## 6. LLM 生成 (LLM Generation)将检索到的上下文 + 用户问题构造为 Prompt，调用大模型生成最终答案。> ⚠️ 此处使用模拟 LLM 响应演示格式。对接真实 LLM 时只需替换 `call_llm()` 函数。

In [ ]:
# @title Prompt 模板与问答函数PROMPT_TEMPLATE = (    "你是一位专业的保险理赔分析师。请根据以下保单上下文，回答用户的问题。\n\n"    "## 上下文（保单片段）\n{context}\n\n"    "## 元数据（保单摘要）\n{metadata}\n\n"    "## 用户问题\n{question}\n\n"    "## 回答要求\n"    "1. 如果上下文明确包含答案，请准确引用原文并说明所在页码。\n"    "2. 如果涉及变更，请明确列出变更前和变更后的内容对比。\n"    "3. 如果上下文不足以回答，请如实说明。\n"    "4. 以中文回答，使用清晰的列表格式。\n\n"    "## 答案")def mock_llm_call(prompt):    pl = prompt.lower()    # Broad/general queries first (check before specific policy numbers in context)    if "哪些保单" in prompt or "从配偶变更" in prompt:        return (            "根据全库检索，以下保单发生过 **受益人从配偶变更为子女** 的变更：\n\n"            "### 1. 保单 P0242025-1883（中国人寿）\n"            "- 变更内容: 李芳（配偶）100% -> 张美玲（女儿）60% + 李芳（配偶）40%\n"            "- 批单号: BG2024-00321 | 变更日期: 2024-06-20\n\n"            "### 2. 保单 TPK-2023-004517（太平洋人寿）\n"            "- 变更内容: 陈静（配偶）50% + 刘伟（父亲）50%"            " -> 陈静（配偶）40% + 刘伟（父亲）30% + 刘晓梅（姐姐）30%\n"            "- 批单号: BG2025-00103 | 变更日期: 2025-01-15\n\n"            "> 共检索到 2 份保单涉及受益人从配偶向子女的变更。"        )    if "p0242025-1883" in pl and ("变更" in prompt or "benef" in pl):        return (            "根据保单 **P0242025-1883**（中国人寿-国寿鑫享金生年金保险 A 款）的档案记录，\n"            "该保单确实发生过 **受益人变更**，详细信息如下：\n\n"            "### 变更批单信息\n"            "- **批单号**: BG2024-00321\n"            "- **变更日期**: 2024-06-20\n"            "- **变更类型**: 受益人变更\n"            "- **变更原因**: 增加子女为共同受益人\n\n"            "### 变更前后对比\n\n"            "| 项目 | 变更前 | 变更后 |\n"            "|------|--------|--------|\n"            "| 受益人 1 | 李芳（配偶）100% | 张美玲（女儿）60% |\n"            "| 受益人 2 | - | 李芳（配偶）40% |\n\n"            "### 所在位置\n"            "- **保单第 4 页** - 《受益人变更批单》（BG2024-00321）\n"            "- **保单第 3 页** - 原始受益人页（变更前为 李芳 100%）"        )    if "tpk-2023-004517" in pl and "变更" in prompt:        return (            "根据保单 **TPK-2023-004517**（太平洋人寿-金佑人生终身寿险）的档案记录，\n"            "该保单共有 **2 次受益人变更记录**，详情如下：\n\n"            "### 第一次变更（BG2024-00892 - 2024-03-10）\n"            "**变更原因**: 增加投保人为共同受益人\n\n"            "| 项目 | 变更前 | 变更后 |\n"            "|------|--------|--------|\n"            "| 受益人 1 | 陈静（配偶）100% | 陈静（配偶）50% |\n"            "| 受益人 2 | - | 刘伟（父亲）50% |\n\n"            "### 第二次变更（BG2025-00103 - 2025-01-15）\n"            "**变更原因**: 增加子女为共同受益人\n\n"            "| 项目 | 变更前 | 变更后 |\n"            "|------|--------|--------|\n"            "| 受益人 1 | 陈静（配偶）50% | 陈静（配偶）40% |\n"            "| 受益人 2 | 刘伟（父亲）50% | 刘伟（父亲）30% |\n"            "| 受益人 3 | - | 刘晓梅（姐姐）30% |\n\n"            "### 所在位置\n"            "- **保单第 3 页** - 第一次受益人变更批单\n"            "- **保单第 4 页** - 第二次受益人变更批单"        )    return "根据提供的保单上下文，未找到与问题直接相关的信息。"def build_context(chunks, parents_list):    sections = []    seen = set()    for chunk, score in chunks:        parent_chunk = next((p for p in parents_list if p.chunk_id == chunk.parent_id), None)        key = str(chunk.page_number) + "_" + (parent_chunk.heading if parent_chunk else "")        if key not in seen:            seen.add(key)            if parent_chunk and parent_chunk.text not in sections:                sections.append("--- Page " + str(chunk.page_number)                                + " | " + parent_chunk.heading + " ---\n"                                + parent_chunk.text)        elif chunk.text not in "\n".join(sections):            sections.append("[Page " + str(chunk.page_number) + " snippet] " + chunk.text)    return "\n".join(sections)def answer_question(question, policy_id=None):    results = hybrid_retrieve(question, policy_id=policy_id, top_k_final=10)    context = build_context(results, parents)    meta = all_metadata.get(policy_id) if policy_id else None    if meta:        ben_str = ", ".join(            b["name"] + "(" + b["relation"] + ") " + b["ratio"]            for b in meta.beneficiaries        )        meta_summary = (            "Policy: " + meta.policy_id + "\n"            "Applicant: " + str(meta.applicant) + " | Insured: " + str(meta.insured) + "\n"            "Product: " + str(meta.product_name) + "\n"            "Beneficiaries: " + ben_str + "\n"            "Endorsements: " + str(len(meta.endorsements))        )    else:        meta_summary = "No specific policy"    prompt = PROMPT_TEMPLATE.format(context=context, metadata=meta_summary, question=question)    return mock_llm_call(prompt), context, meta_summaryprint("Q&A engine ready")

---## 7. 端到端演示 (End-to-End Demo)### 场景 A：查询保单 P0242025-1883 的受益人变更情况

In [ ]:
# @title 场景 A：指定保单号的受益人变更查询q = '保单 P0242025-1883 的受益人是否有过变更？变更内容是什么？'answer, context, meta = answer_question(q, policy_id='P0242025-1883')print('='*60); print('  Question'); print('='*60); print(f'  {q}\n')print('='*60); print('  Policy Metadata'); print('='*60); print(f'  {meta}\n')print('='*60); print('  Retrieved Context'); print('='*60)ctx = context[:600] + '...' if len(context) > 600 else context; print(f'{ctx}\n')print('='*60); print('  LLM Answer'); print('='*60); print(answer)

### 场景 B：查询保单 TPK-2023-004517 的受益人变更（多次变更历史）

In [ ]:
# @title 场景 B：多次变更历史的保单q = 'TPK-2023-004517 的受益人变更历史是怎样的？'answer, context, meta = answer_question(q, policy_id='TPK-2023-004517')print('='*60); print('  Question'); print('='*60); print(f'  {q}\n')print('='*60); print('  Policy Metadata'); print('='*60); print(f'  {meta}\n')print('='*60); print('  LLM Answer'); print('='*60); print(answer)

### 场景 C：模糊查询（不指定保单号，全库检索）

In [ ]:
# @title 场景 C：全库模糊搜索q = '哪些保单的受益人从配偶变更为了子女？'answer, context, meta = answer_question(q, policy_id=None)print('='*60); print('  Question'); print('='*60); print(f'  {q}\n')print('='*60); print('  LLM Answer'); print('='*60); print(answer)

---## 8. 检索质量评估 (Retrieval Quality Evaluation)定义 3 个测试用例，评估系统是否能正确召回包含受益人变更内容的页面。

In [ ]:
# @title 评估：召回率与命中位置test_cases = [    {'question': 'P0242025-1883 张美玲 60% 受益人 变更', 'policy_id': 'P0242025-1883', 'expected_page': 4, 'expected_keywords': ['张美玲', '60%', '变更']},    {'question': 'TPK-2023-004517 刘晓梅 姐姐 受益人 30%', 'policy_id': 'TPK-2023-004517', 'expected_page': 4, 'expected_keywords': ['刘晓梅', '30%', '姐姐']},    {'question': 'P2024-66892 赵强 100% 配偶 受益人', 'policy_id': 'P2024-66892', 'expected_page': 3, 'expected_keywords': ['赵强', '100%', '配偶']},]results = []for tc in test_cases:    hits = hybrid_retrieve(tc['question'], policy_id=tc['policy_id'], top_k_final=5)    hit_pages = [c.page_number for c, _ in hits]    hit_texts = [c.text for c, _ in hits]    page_hit = tc['expected_page'] in hit_pages    all_text = ' '.join(hit_texts)    kw_hits = sum(1 for kw in tc['expected_keywords'] if kw in all_text)    results.append({'policy': tc['policy_id'], 'page_hit': 'OK' if page_hit else 'MISS', 'pages': str(hit_pages[:3]), 'kw_recall': f'{kw_hits}/{len(tc["expected_keywords"])}', 'top1': f'p{hit_pages[0]}' if hit_pages else '-'})print(tabulate(results, headers='keys', tablefmt='grid'))print(f'\nPage hit rate: {sum(1 for r in results if r["page_hit"] == "OK")}/{len(results)}')

---## 9. 总结与生产化建议### PoC 验证结果 ✅| 能力 | 状态 ||------|------|| 多页 TIF 解析（Mock OCR + Markdown） | ✅ 模拟完成 || 元数据提取（保单号/投保人/受益人） | ✅ 正则提取 || 父子分块（Parent-Child Chunking） | ✅ 层级化分块 || 向量检索（Dense） | ✅ Sentence-BERT || 全文检索（BM25） | ✅ jieba + BM25 || RRF 融合 + Metadata 过滤 | ✅ 混合检索 || LLM 生成（Prompt + 上下文） | ✅ Mock 演示 |### 🔧 生产化建议| 模块 | 生产方案 | 说明 ||------|----------|------|| **OCR** | PP-OCRv4 / LayoutLMv3 | 中文保单专用模型，支持表格结构识别 || **向量模型** | BAAI/bge-large-zh-v1.5 | 中文 embeddings，支持 Matryoshka 量化 || **向量数据库** | Milvus / Qdrant / pgvector | 支持标量过滤 + 向量检索混合查询 || **BM25** | Elasticsearch 8.x | 中文 IK 分词 + BM25 全文索引 || **Reranker** | BAAI/bge-reranker-v2-m3 | 交叉编码器，对 Top-100 做精排 || **LLM** | DeepSeek / Qwen / GPT-4o | 需配合保险领域 Prompt || **Pipeline** | 阿里云 FaaS / Airflow | TIF 上传 OSS 触发异步 ETL |### 💡 下一步1. 用真实 TIF 保单替换 Mock 数据2. 接入真实 OCR 引擎提取文本3. 部署向量数据库与搜索服务4. 构建前端 Demo 页面（Streamlit / Gradio）5. 建立标注数据集评估检索精度（MRR / NDCG）